# Simulating a framed MaMMoS anisotropy pattern

This notebook embeds the **MaMMoS** word mark as an anisotropy pattern in a continuous thin film and surrounds that film with a one-grain-wide frame. Every frame grain is a cube whose edge length equals the film thickness. The grains form a clockwise closed curve. A material-level switch selects whether their uniaxial easy axes perform a transverse rotation around that curve or simply follow its clockwise tangent with a fixed downward tilt. The mesh is shared by both variants. The remaining setup follows the standalone word-mark simulation, including a minimal four-point field sweep to remanence.

In [1]:
import warnings

import micromagneticmodel as mm
import numpy as np
import pyvista as pv
import tetgen
from matplotlib.colors import LinearSegmentedColormap

import tommos_adapter as ta

pv.set_jupyter_backend("server")

## Define the word mark and thin-film geometry

PyVista creates the word mark in the $xy$ plane and extrudes it along $z$. The coordinates below are mesh units representing nanometres. A height of 40 and a depth of 10 therefore correspond to $40\,\mathrm{nm}$ and $10\,\mathrm{nm}$. The central film dimensions are rounded up to multiples of the film thickness. A one-grain-wide outer frame can therefore be tiled completely with equal $10\times10\times10\,\mathrm{nm}^3$ cubes.

In [2]:
wordmark_surface = (
    pv.Text3D(
        "MaMMoS",
        height=40.0,
        depth=10.0,
        center=(0, 0, 0),
    )
    .triangulate()
    .clean()
    .compute_normals(auto_orient_normals=True, consistent_normals=True)
)

if wordmark_surface.n_open_edges != 0:
    raise RuntimeError(f"The word-mark surface has {wordmark_surface.n_open_edges} open edges.")

film_margin = 10.0  # minimum nm around the word mark in the xy plane
film_thickness = wordmark_surface.bounds.z_max - wordmark_surface.bounds.z_min
grain_size = film_thickness
film_width = grain_size * np.ceil(
    (wordmark_surface.bounds.x_max - wordmark_surface.bounds.x_min + 2 * film_margin) / grain_size
)
film_height = grain_size * np.ceil(
    (wordmark_surface.bounds.y_max - wordmark_surface.bounds.y_min + 2 * film_margin) / grain_size
)
film_center_x = 0.5 * (wordmark_surface.bounds.x_min + wordmark_surface.bounds.x_max)
film_center_y = 0.5 * (wordmark_surface.bounds.y_min + wordmark_surface.bounds.y_max)
film_bounds = (
    film_center_x - 0.5 * film_width,
    film_center_x + 0.5 * film_width,
    film_center_y - 0.5 * film_height,
    film_center_y + 0.5 * film_height,
    wordmark_surface.bounds.z_min,
    wordmark_surface.bounds.z_max,
)
film_surface = pv.Box(bounds=film_bounds).triangulate().clean()

framed_film_bounds = (
    film_bounds[0] - grain_size,
    film_bounds[1] + grain_size,
    film_bounds[2] - grain_size,
    film_bounds[3] + grain_size,
    film_bounds[4],
    film_bounds[5],
)
framed_film_surface = pv.Box(bounds=framed_film_bounds).triangulate().clean()

wordmark_surface

PolyData (0x31c2d6620)
  N Cells:    1028
  N Points:   522
  N Strips:   0
  X Bounds:   -1.161e+02, 1.161e+02
  Y Bounds:   -2.000e+01, 2.000e+01
  Z Bounds:   -5.000e+00, 5.000e+00
  N Arrays:   2

## Generate and inspect the tetrahedral mesh

TetGen fills the framed film with linear tetrahedra. `max_tetrahedron_volume` is expressed in cubic mesh units, here $\mathrm{nm}^3$: smaller values both refine the film mesh and improve the cell-wise approximation of the word-mark boundary. After meshing, each tetrahedron is assigned to the central `wordmark` or `thinfilm` region or to one of the equal cubic frame grains according to its centre. Frame grains are ordered clockwise, starting at the upper-left corner.

In [3]:
max_tetrahedron_volume = 2.0  # nm^3; decrease this value for a finer mesh

with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message="`PolyData._connectivity_array` is deprecated.*",
        category=pv.PyVistaDeprecationWarning,
    )
    tet = tetgen.TetGen(framed_film_surface)
    tet.tetrahedralize(
        order=1,
        minratio=3.0,
        fixedvolume=True,
        maxvolume=max_tetrahedron_volume,
    )
mesh = tet.grid

if not np.all(mesh.celltypes == pv.CellType.TETRA):
    raise RuntimeError("TetGen generated non-tetrahedral cells.")

cell_centres = mesh.cell_centers()
inside_wordmark = np.asarray(
    cell_centres.select_interior_points(
        wordmark_surface,
        method="signed_distance",
        check_surface=True,
    ).point_data["selected_points"]
).astype(bool)

centre_points = np.asarray(cell_centres.points)
n_grains_x = int(round((framed_film_bounds[1] - framed_film_bounds[0]) / grain_size))
n_grains_y = int(round((framed_film_bounds[3] - framed_film_bounds[2]) / grain_size))
frame_grid_indices = (
    [(i, n_grains_y - 1) for i in range(n_grains_x)]
    + [(n_grains_x - 1, j) for j in range(n_grains_y - 2, -1, -1)]
    + [(i, 0) for i in range(n_grains_x - 2, -1, -1)]
    + [(0, j) for j in range(1, n_grains_y - 1)]
)
frame_region_names = [f"frame_{i:03d}" for i in range(len(frame_grid_indices))]
material_regions = {
    "thinfilm": 1,
    "wordmark": 2,
    **{name: i + 3 for i, name in enumerate(frame_region_names)},
}

frame_region_lookup = np.full((n_grains_y, n_grains_x), -1, dtype=np.int32)
for frame_index, (grid_x, grid_y) in enumerate(frame_grid_indices):
    frame_region_lookup[grid_y, grid_x] = frame_index

cell_grid_x = np.floor((centre_points[:, 0] - framed_film_bounds[0]) / grain_size).astype(int)
cell_grid_y = np.floor((centre_points[:, 1] - framed_film_bounds[2]) / grain_size).astype(int)
cell_grid_x = np.clip(cell_grid_x, 0, n_grains_x - 1)
cell_grid_y = np.clip(cell_grid_y, 0, n_grains_y - 1)
cell_frame_indices = frame_region_lookup[cell_grid_y, cell_grid_x]

cell_region_ids = np.full(
    mesh.n_cells,
    material_regions["thinfilm"],
    dtype=np.int32,
)
is_frame = cell_frame_indices >= 0
cell_region_ids[is_frame] = cell_frame_indices[is_frame] + 3
cell_region_ids[inside_wordmark] = material_regions["wordmark"]
mesh.cell_data["region_id"] = cell_region_ids

frame_grain_centres = np.array(
    [
        (
            framed_film_bounds[0] + (grid_x + 0.5) * grain_size,
            framed_film_bounds[2] + (grid_y + 0.5) * grain_size,
            0.0,
        )
        for grid_x, grid_y in frame_grid_indices
    ]
)

if set(np.unique(cell_region_ids)) != set(material_regions.values()):
    raise RuntimeError("The mesh does not contain all material regions.")

np.testing.assert_allclose(mesh.center, (0, 0, 0), atol=1e-12)
print(f"Mesh points: {mesh.n_points}")
print(f"Tetrahedra: {mesh.n_cells}")
print(f"Frame grains: {len(frame_region_names)}")
mesh

Mesh points: 39826
Tetrahedra: 201795
Frame grains: 68


UnstructuredGrid (0x31c2d6e60)
  N Cells:    201795
  N Points:   39826
  X Bounds:   -1.400e+02, 1.400e+02
  Y Bounds:   -4.000e+01, 4.000e+01
  Z Bounds:   -5.000e+00, 5.000e+00
  N Arrays:   1

In [4]:
mesh_plotter = pv.Plotter()
mesh_plotter.add_mesh(
    mesh,
    scalars="region_id",
    categories=True,
    cmap="twilight",
    show_edges=True,
    show_scalar_bar=False,
)
mesh_plotter.view_vector((0.1, -0.1, 1.0), viewup=(0, 1, 0))
mesh_plotter.show(jupyter_backend="server")

Widget(value='<iframe src="http://localhost:51439/index.html?ui=P_0x10903bd10_0&reconnect=auto" class="pyvista…

## Define the multi-grain micromagnetic model

The visualization mesh remains in nanometres. A deep copy is converted to SI units for the micromagnetic model. The central film has a negative uniaxial constant with $u=z$, which makes the $xy$ plane easy, whereas the MaMMoS word mark has a positive constant and an easy axis along $z$. Every cubic frame grain has a positive uniaxial constant. Set `frame_anisotropy_mode` to `"transverse"` for rotations in the local plane spanned by the outward normal and $-z$, or to `"clockwise"` for axes that follow the clockwise curve tangent with a fixed $20^\circ$ downward tilt. `frame_transverse_turns` and `frame_phase_offset` only affect the transverse mode. Because uniaxial axes identify $u$ and $-u$, a $360^\circ$ vector rotation corresponds to two full director rotations. The initial unit magnetization remains aligned with $z$.

In [5]:
# TODO: Use permalloy material parameters from the ab-initio database.
Ms = 8e5  # saturation magnetization in A/m
A = 7e-12  # exchange stiffness in J/m
frame_anisotropy = 1e7  # J/m^3
K = {
    "thinfilm": -1e7,
    "wordmark": 1e7,
    **{name: frame_anisotropy for name in frame_region_names},
}

previous_centres = np.roll(frame_grain_centres, 1, axis=0)
next_centres = np.roll(frame_grain_centres, -1, axis=0)
frame_tangents = next_centres - previous_centres
frame_tangents /= np.linalg.norm(frame_tangents, axis=1)[:, np.newaxis]
frame_outward_normals = np.column_stack((-frame_tangents[:, 1], frame_tangents[:, 0], np.zeros(len(frame_tangents))))

frame_anisotropy_mode = "transverse"  # "transverse" or "clockwise"
frame_transverse_turns = 3.0
frame_phase_offset = np.deg2rad(20)
frame_curve_fraction = np.arange(len(frame_region_names)) / len(frame_region_names)
frame_down = np.array((0.0, 0.0, -1.0))

if frame_anisotropy_mode == "transverse":
    frame_phases = frame_phase_offset + 2 * np.pi * frame_transverse_turns * frame_curve_fraction
    frame_axes = (
        np.cos(frame_phases)[:, np.newaxis] * frame_outward_normals + np.sin(frame_phases)[:, np.newaxis] * frame_down
    )
elif frame_anisotropy_mode == "clockwise":
    frame_axes = np.cos(frame_phase_offset) * frame_tangents + np.sin(frame_phase_offset) * frame_down
else:
    raise ValueError("frame_anisotropy_mode must be 'transverse' or 'clockwise'.")

u = {
    "thinfilm": (0, 0, 1),
    "wordmark": (0, 0, 1),
    **{name: tuple(axis) for name, axis in zip(frame_region_names, frame_axes, strict=True)},
}

### Inspect the selected frame anisotropy

One centered arrow is drawn at each frame-grain centre and colored by its $z$ component. In `transverse` mode the arrows rotate in the plane perpendicular to the curve tangent; in `clockwise` mode they follow the tangent with a constant downward tilt. This plot shows the selected anisotropy field before any micromagnetic calculation is run.

In [6]:
# NBVAL_SKIP
anisotropy_points = pv.PolyData(frame_grain_centres)
anisotropy_points.point_data["u"] = frame_axes
anisotropy_points.point_data["u_z"] = frame_axes[:, 2]

centered_anisotropy_arrow = pv.Arrow(start=(-0.5, 0, 0), direction=(1, 0, 0))
anisotropy_glyphs = anisotropy_points.glyph(
    orient="u",
    scale=False,
    factor=0.8 * grain_size,
    geom=centered_anisotropy_arrow,
)

anisotropy_plotter = pv.Plotter()
anisotropy_plotter.add_mesh(
    mesh.extract_surface(algorithm="dataset_surface"),
    scalars="region_id",
    cmap="twilight",
    opacity=0.2,
    show_scalar_bar=False,
)
anisotropy_plotter.add_mesh(
    anisotropy_glyphs,
    scalars="u_z",
    cmap="coolwarm",
    clim=(-1, 1),
    scalar_bar_args={"title": "anisotropy u_z"},
)
anisotropy_plotter.add_text(
    f"mode: {frame_anisotropy_mode}",
    position="upper_left",
)
anisotropy_plotter.add_axes()
anisotropy_plotter.view_vector((0.1, -0.1, 1.0), viewup=(0, 1, 0))
anisotropy_plotter.show(jupyter_backend="server")

Widget(value='<iframe src="http://localhost:51439/index.html?ui=P_0x1089b2350_1&reconnect=auto" class="pyvista…

In [7]:
mesh_si = mesh.copy(deep=True)
mesh_si.points *= ta.scripts.DEFAULT_MESH_UNIT

mesh_si.field_data["region_ids"] = np.array(
    list(material_regions.values()),
    dtype=np.int32,
)
mesh_si.field_data["region_names"] = list(material_regions)
mesh_si.cell_data["Ms"] = np.full(mesh_si.n_cells, Ms)
mesh_si.point_data["m"] = np.tile((0.0, 0.0, 1.0), (mesh_si.n_points, 1))

mesh_si

UnstructuredGrid (0x3210839a0)
  N Cells:    201795
  N Points:   39826
  X Bounds:   -1.400e-07, 1.400e-07
  Y Bounds:   -4.000e-08, 4.000e-08
  Z Bounds:   -5.000e-09, 5.000e-09
  N Arrays:   5

The model contains uniform exchange, region-dependent uniaxial anisotropy, and demagnetization energy. The anisotropy dictionaries are resolved through the material names stored in the mesh. These illustrative parameters can be refined before running a physically resolved calculation.

In [8]:
system = mm.System(name=f"mammos_{frame_anisotropy_mode}_frame", m=mesh_si)
system.energy = mm.Exchange(A=A) + mm.UniaxialAnisotropy(K=K, u=u) + mm.Demag()
system.T = 0

system

System(name='mammos_transverse_frame')

## Run a minimal hysteresis calculation

The applied field is almost parallel to $z$: it follows the word-mark easy axis and is normal to the central film's easy plane. Its angle to the frame axes varies with the transverse phase in `transverse` mode; in `clockwise` mode it has a constant acute angle of $70^\circ$ to the frame axes. The field decreases from $+2\,\mathrm{T}$ to $-0.2\,\mathrm{T}$. Four field points keep this run short; they demonstrate the complete workflow but are not sufficient for a physically resolved reversal curve.

In [9]:
# TODO: switch to energy minimization (mindriver) instead of running a hysteresis loop
field_limit = 2.0 / mm.consts.mu0
driver = ta.HysteresisDriver()
driver.drive(
    system,
    h_direction=(0, 0.001, 1),
    h_start=field_limit,
    h_final=-0.1 * field_limit,
    n=4,
)

Running Tommos (TommosRunner)[2026-11-13T14:12:30]... (42.1 s)


## Plot the final magnetization without the Airshell

After the drive, the adapter maps the final nodal magnetization back onto the original PyVista mesh. Consequently, `system.m` contains the framed, patterned thin film but not the auxiliary tommos Airshell. The glyphs below are centered on their mesh points and show the final magnetization vectors, colored by their $z$ component.

In [11]:
# NBVAL_SKIP
magnetic_state = system.m.extract_cells(np.asarray(system.m.cell_data["Ms"]) > 0)
magnetic_state.point_data["m_x"] = np.asarray(magnetic_state.point_data["m"])[:, 0]
magnetic_state.point_data["m_y"] = np.asarray(magnetic_state.point_data["m"])[:, 1]
magnetic_state.point_data["m_z"] = np.asarray(magnetic_state.point_data["m"])[:, 2]

domain_lengths = np.array(
    [
        magnetic_state.bounds[1] - magnetic_state.bounds[0],
        magnetic_state.bounds[3] - magnetic_state.bounds[2],
        magnetic_state.bounds[5] - magnetic_state.bounds[4],
    ]
)
glyph_length = 0.05 * domain_lengths.max()

centered_arrow = pv.Arrow(start=(-0.5, 0, 0), direction=(1, 0, 0))
magnetization_glyphs = magnetic_state.glyph(
    orient="m",
    scale=False,
    factor=glyph_length,
    geom=centered_arrow,
    tolerance=None,
)

mammos_cmap = LinearSegmentedColormap.from_list("mammos", ["#62B7CF", "#F7F7F7", "#2CB877"])

vector_plotter = pv.Plotter()
vector_plotter.add_mesh(
    magnetic_state.extract_surface(algorithm="dataset_surface"),
    scalars="region_id",
    categories=True,
    opacity=0.15,
    show_scalar_bar=False,
)
vector_plotter.add_mesh(
    magnetization_glyphs,
    scalars="m_z",
    cmap=mammos_cmap,
    clim=(0.0, 1),
    scalar_bar_args={"title": "m_z"},
)
vector_plotter.add_axes()
# Show the xy plane almost face-on while retaining a slight three-dimensional tilt.
vector_plotter.view_vector((0.1, -0.1, 1.0), viewup=(0, 1, 0))
vector_plotter.show(jupyter_backend="server")

Widget(value='<iframe src="http://localhost:51439/index.html?ui=P_0x328877a10_3&reconnect=auto" class="pyvista…